In [11]:
"""
Data Collection Script
======================
Run this script ONCE on your local machine to fetch all raw data.
Outputs are saved to the /data folder and committed to GitHub.
The main analysis notebook reads from those CSV files directly.

Requirements:
    pip install requests pandas

NOTE: If running inside Jupyter, replace DATA_DIR with:
    DATA_DIR = Path("../data")
"""

import requests
import pandas as pd
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
# Use Path("../data") if running inside Jupyter
DATA_DIR = Path("data").parent / "data"
DATA_DIR.mkdir(exist_ok=True)

# ── Time range ─────────────────────────────────────────────────────────────────
START_DATE = "2015-01-01"
END_DATE   = "2025-12-31"


# ══════════════════════════════════════════════════════════════════════════════
# SOURCE 2: Open-Meteo — Hourly Weather for Five Bulgarian Cities
# ══════════════════════════════════════════════════════════════════════════════

# Five largest Bulgarian cities by population.
# Population-proportional weighting is performed in the analysis notebook,
# where the methodology is documented and explained explicitly.

CITIES = [
    {"name": "Sofia",   "lat": 42.6977, "lon": 23.3219},
    {"name": "Plovdiv", "lat": 42.1354, "lon": 24.7453},
    {"name": "Varna",   "lat": 43.2141, "lon": 27.9147},
    {"name": "Burgas",  "lat": 42.5048, "lon": 27.4626},
    {"name": "Ruse",    "lat": 43.8356, "lon": 25.9657},
]

WEATHER_VARIABLES = "temperature_2m,precipitation,windspeed_10m,cloudcover,snowfall"


def fetch_city_weather(city: dict) -> pd.DataFrame:
    """
    Fetches hourly weather for a single city from Open-Meteo Historical API.
    Returns a DataFrame with daily aggregated values.

    Aggregation method per variable:
        - temperature_2m  : daily mean
        - precipitation   : daily sum
        - windspeed_10m   : daily mean
        - cloudcover      : daily mean
        - snowfall        : daily sum
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   city["lat"],
        "longitude":  city["lon"],
        "start_date": START_DATE,
        "end_date":   END_DATE,
        "hourly":     WEATHER_VARIABLES,
        "timezone":   "Europe/Sofia",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame({
        "time":             data["hourly"]["time"],
        "temperature_c":    data["hourly"]["temperature_2m"],
        "precipitation_mm": data["hourly"]["precipitation"],
        "windspeed_kmh":    data["hourly"]["windspeed_10m"],
        "cloudcover_pct":   data["hourly"]["cloudcover"],
        "snowfall_cm":      data["hourly"]["snowfall"],
    })
    df["time"] = pd.to_datetime(df["time"])
    df["date"] = df["time"].dt.date

    # Aggregate hourly → daily
    daily = df.groupby("date").agg(
        temperature_c    = ("temperature_c",    "mean"),
        precipitation_mm = ("precipitation_mm", "sum"),
        windspeed_kmh    = ("windspeed_kmh",    "mean"),
        cloudcover_pct   = ("cloudcover_pct",   "mean"),
        snowfall_cm      = ("snowfall_cm",      "sum"),
    ).reset_index()

    daily["date"] = pd.to_datetime(daily["date"])
    return daily


def fetch_weather() -> None:
    """
    Fetches daily weather data for five Bulgarian cities and saves
    each as a separate CSV file.

    Population-weighted averaging is performed in the analysis notebook.
    """
    print("Fetching weather data from Open-Meteo for 5 Bulgarian cities...")

    for city in CITIES:
        print(f"  Fetching {city['name']}...")
        df = fetch_city_weather(city)
        filename = f"weather_{city['name'].lower()}.csv"
        df.to_csv(DATA_DIR / filename, index=False)
        print(f"    Saved {filename} — {len(df):,} rows")

    print("\nDone. Individual city weather files saved to /data.")
    print("\nRemaining steps:")
    print("  1. Filter Bulgaria from Ember CSV and save to data/ember_prices_bulgaria.csv")
    print("  2. Download Eurostat HICP for Bulgaria and save to data/eurostat_hicp_bulgaria.csv")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    fetch_weather()

Fetching weather data from Open-Meteo for 5 Bulgarian cities...
  Fetching Sofia...
    Saved weather_sofia.csv — 4,018 rows
  Fetching Plovdiv...
    Saved weather_plovdiv.csv — 4,018 rows
  Fetching Varna...
    Saved weather_varna.csv — 4,018 rows
  Fetching Burgas...
    Saved weather_burgas.csv — 4,018 rows
  Fetching Ruse...
    Saved weather_ruse.csv — 4,018 rows

Done. Individual city weather files saved to /data.

Remaining steps:
  1. Filter Bulgaria from Ember CSV and save to data/ember_prices_bulgaria.csv
  2. Download Eurostat HICP for Bulgaria and save to data/eurostat_hicp_bulgaria.csv
